In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from data.combine_timetable import combine_timetables
from data.timetable import load_gold, save_gold


In [2]:
n_freight_trains = 343

passenger = load_gold('passenger')
freight   = load_gold('freight', n_trains=n_freight_trains)

# Validatie
overlap = set(passenger['TRAIN_NO']) & set(freight['TRAIN_NO'])
print(f"Overlappende TRAIN_NO: {overlap if overlap else 'geen'}")

combined = combine_timetables(n_freight_trains)
print(combined['TRAIN_TYPE'].value_counts())
print(f"ENTRY_SECONDS range: {combined['ENTRY_SECONDS'].min():.0f}s — {combined['ENTRY_SECONDS'].max():.0f}s")

Overlappende TRAIN_NO: geen
TRAIN_TYPE
IC         6158
L          3275
freight    1239
EURST       324
INT         189
ICE         126
Name: count, dtype: int64
ENTRY_SECONDS range: 0s — 87080s


In [3]:
combined.head()

,SECTION,SOURCE,TARGET,PLANNED_ENTRY,PLANNED_EXIT,TRAIN_NO,TYPE,TRAIN_TYPE,PERIOD,DYNAMICS,ENTRY_SECONDS,EXIT_SECONDS
0,36N:SCHAARBEEK-BRUSSEL-NOORD,SCHAARBEEK,BRUSSEL-NOORD,2025-01-01 21:23:00,2025-01-01 21:26:00,10,SOURCE,ICE,EVENING,ACC-BR,76863,77043
1,BRUSSEL-NOORD -- platform 3,BRUSSEL-NOORD,BRUSSEL-NOORD,2025-01-01 21:26:00,2025-01-01 21:28:00,10,WITHIN-STATION-DWELL,ICE,EVENING,0-0,77043,77163
2,0-2:BRUSSEL-NOORD-BRUSSEL-CONGRES,BRUSSEL-NOORD,BRUSSEL-CONGRES,2025-01-01 21:28:00,2025-01-01 21:30:00,10,BETWEEN-STATION,ICE,EVENING,ACC-0,77163,77283
3,BRUSSEL-CONGRES -- platform 1,BRUSSEL-CONGRES,BRUSSEL-CONGRES,2025-01-01 21:30:00,2025-01-01 21:30:30,10,WITHIN-STATION-PASSING,ICE,EVENING,0-0,77283,77313
4,0-2:BRUSSEL-CONGRES-BRUSSEL-CENTRAAL,BRUSSEL-CONGRES,BRUSSEL-CENTRAAL,2025-01-01 21:30:30,2025-01-01 21:31:00,10,BETWEEN-STATION,ICE,EVENING,0-0,77313,77343


In [4]:
save_gold(combined, source='combined', n_trains=n_freight_trains)

Opgeslagen (combined): 1371 treinen, 11311 segmenten → /Users/ddw/Desktop/Rescheduling/data/gold/combined/343


In [3]:
import sys
sys.path.insert(0, "/Users/ddw/Desktop/Rescheduling")

import pandas as pd
from data.timetable import load_gold

# Laad combined timetable
df = load_gold('combined', n_trains=182)

# Inspecteer de twee conflicterende treinen
train_ids = [2076, 3228]

for tid in train_ids:
    subset = df[df['TRAIN_NO'] == tid].copy()
    subset = subset.sort_values('ENTRY_SECONDS')
    print(f"\n{'='*70}")
    print(f"TREIN {tid} — {len(subset)} segmenten")
    print(f"{'='*70}")
    print(subset[[
        'SECTION', 'SOURCE', 'TARGET', 'TYPE',
        'PLANNED_ENTRY', 'PLANNED_EXIT',
        'ENTRY_SECONDS', 'EXIT_SECONDS',
        'TRAIN_TYPE', 'DYNAMICS'
    ]].to_string(index=False))

# Welke segmenten delen ze?
segs_2076 = set(df[df['TRAIN_NO'] == 2076]['SECTION'])
segs_3228 = set(df[df['TRAIN_NO'] == 3228]['SECTION'])
shared = segs_2076 & segs_3228

print(f"\n{'='*70}")
print(f"Gedeelde segmenten: {shared}")

if shared:
    print(f"\nTijden op gedeelde segmenten:")
    for seg in sorted(shared):
        row_2076 = df[(df['TRAIN_NO'] == 2076) & (df['SECTION'] == seg)]
        row_3228 = df[(df['TRAIN_NO'] == 3228) & (df['SECTION'] == seg)]
        if not row_2076.empty and not row_3228.empty:
            print(f"\n  Segment: {seg}")
            print(f"  2076: entry={row_2076['ENTRY_SECONDS'].values[0]}s "
                  f"exit={row_2076['EXIT_SECONDS'].values[0]}s")
            print(f"  3228: entry={row_3228['ENTRY_SECONDS'].values[0]}s "
                  f"exit={row_3228['EXIT_SECONDS'].values[0]}s")
            diff = abs(row_2076['ENTRY_SECONDS'].values[0] - row_3228['ENTRY_SECONDS'].values[0])
            print(f"  Verschil entry: {diff}s")

# Check ook het blokkerende segment
print(f"\n{'='*70}")
print(f"Segment '50:JETTE-BOCKSTAEL' in timetable van 3228:")
seg_info = df[(df['TRAIN_NO'] == 3228) & (df['SECTION'] == '50:JETTE-BOCKSTAEL')]
if not seg_info.empty:
    print(seg_info[['SECTION', 'TYPE', 'ENTRY_SECONDS', 'EXIT_SECONDS']].to_string(index=False))
else:
    print("  Niet gevonden in timetable van 3228")

print(f"\nSegment '50:JETTE-BOCKSTAEL' in timetable van 2076:")
seg_info = df[(df['TRAIN_NO'] == 2076) & (df['SECTION'] == '50:JETTE-BOCKSTAEL')]
if not seg_info.empty:
    print(seg_info[['SECTION', 'TYPE', 'ENTRY_SECONDS', 'EXIT_SECONDS']].to_string(index=False))
else:
    print("  Niet gevonden in timetable van 2076")


TREIN 2076 — 13 segmenten
                                 SECTION              SOURCE              TARGET                   TYPE       PLANNED_ENTRY        PLANNED_EXIT  ENTRY_SECONDS  EXIT_SECONDS TRAIN_TYPE DYNAMICS
            50:SINT-AGATHA-BERCHEM-JETTE SINT-AGATHA-BERCHEM               JETTE                 SOURCE 2025-01-01 06:35:00 2025-01-01 06:38:00          23583         23763          L   ACC-BR
                     JETTE -- platform 1               JETTE               JETTE   WITHIN-STATION-DWELL 2025-01-01 06:38:00 2025-01-01 06:39:00          23763         23823          L      0-0
                      50:JETTE-BOCKSTAEL               JETTE           BOCKSTAEL        BETWEEN-STATION 2025-01-01 06:39:00 2025-01-01 06:41:00          23823         23943          L   ACC-BR
                 BOCKSTAEL -- platform 1           BOCKSTAEL           BOCKSTAEL   WITHIN-STATION-DWELL 2025-01-01 06:41:00 2025-01-01 06:43:00          23943         24063          L      0-0
        